In [ ]:
from torch.optim import Optimizer
from torch.optim.lr_scheduler import (
    LinearLR,
    CosineAnnealingLR,
    CosineAnnealingWarmRestarts,
    SequentialLR,
    LRScheduler,
)



def get_scheduler(
    optimizer: Optimizer, train_dataloader, type) -> LRScheduler:

    steps_per_epoch = len(train_dataloader)
    total_steps = 19
    warmup_steps = 5
    scheduler_type = type

    #no restart with just warmup
    if scheduler_type == "cosine_warmup":

        warmup = LinearLR(
            optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)

        cosine = CosineAnnealingLR(
            optimizer, T_max=total_steps - warmup_steps)
        return SequentialLR( #combine into seq
            optimizer,
            schedulers=[warmup, cosine],
            milestones=[warmup_steps])

    #linear + simple cosine annealing with restart
    elif scheduler_type == "cosine_warmup_restarts":
        warmup = LinearLR(optimizer, start_factor=0.01,
            end_factor=1.0, total_iters=warmup_steps,)

        cosine = CosineAnnealingWarmRestarts(
            optimizer, T_0=total_steps - warmup_steps, T_mult=1, eta_min=0.0)

        return SequentialLR(
            optimizer,
            schedulers=[warmup, cosine],
            milestones=[warmup_steps],
        )
    else:
        raise ValueError(
            f"Unsupported scheduler: {scheduler_type}")


In [3]:
import torch

In [7]:
model = torch.nn.Linear(10, 1)
train_dataloader = [torch.rand(10) for _ in range(20)]  # Dummy dataloader



optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
sc = get_scheduler(optimizer, train_dataloader, "cosine_warmup")


sc.state_dict

<bound method SequentialLR.state_dict of <torch.optim.lr_scheduler.SequentialLR object at 0x0000018886B929F0>>